# ODIN SN2 – Ring-down analysis

Analyze Moku Phasemeter recordings from ODIN session SN2:

- `data/ODIN/20260511_SN2.csv.zip`
- `data/ODIN/20260512_SN2.csv.zip`

Each file contains **two ring-down time series**: differential phase on inputs 1−2 (`tm1_cycles`) and inputs 3−4 (`tm2_cycles`), as in the EDU Day 02 notebook.


In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from mokutools.phasemeter import MokuPhasemeterObject
from ringdownanalysis import RingDownAnalyzer, configure_logging, plots
from ringdownanalysis.estimators import DFTFrequencyEstimator

plots.apply_plotting_style()
plt.style.use("default")
%matplotlib inline

configure_logging(level=logging.WARNING)


In [ ]:
# --- paths and acquisition windows ---
data_dir = Path("../data/ODIN")

# start_time matches the first timestamp in each CSV (seconds in file)
RINGDOWN_FILES = [
    {
        "name": "20260511_SN2.csv.zip",
        "label": "2026-05-11 SN2",
        "start_time": 350.0,
    },
    {
        "name": "20260512_SN2.csv.zip",
        "label": "2026-05-12 SN2",
        "start_time": 150.0,
    },
]

# Load the strong ring-down segment at the start of each recording (~4 h)
ANALYSIS_DURATION_S = 4 * 3600

TAU_INIT = 4000.0
RINGDOWN_CHANNELS = [
    ("tm1_cycles", "1_cycles", "2_cycles"),
    ("tm2_cycles", "3_cycles", "4_cycles"),
]

dft_est = DFTFrequencyEstimator(window="rect", f_min=1.0)
analyzer = RingDownAnalyzer(dft_estimator=dft_est)

NOISY_FIT_PARAMS = dict(
    initial_params=None,
    tau_init=TAU_INIT,
    max_nfev=1500,
    ftol=1e-7,
    xtol=1e-7,
    gtol=1e-7,
    max_tau_multiplier=1,
)

for entry in RINGDOWN_FILES:
    path = data_dir / entry["name"]
    if not path.exists():
        raise FileNotFoundError(f"Missing data file: {path.resolve()}")
    print(f"OK: {path.name}")


In [ ]:
def add_differential_channels(df):
    df["tm1_cycles"] = df["1_cycles"] - df["2_cycles"]
    df["tm2_cycles"] = df["3_cycles"] - df["4_cycles"]
    return df


def format_result_summary(r):
    tau_profile = r.get("tau_profile")
    tau_str = f"tau_profile={tau_profile:.1f} s" if tau_profile is not None else "tau=N/A"
    q_profile = r.get("Q_profile")
    q_prof_str = f"{q_profile:.2e}" if q_profile is not None else "N/A"
    q_nls = r.get("Q_nls")
    q_nls_str = f"{q_nls:.2e}" if q_nls is not None else "N/A"
    return (
        f"f_nls={r['f_nls']:.6f} Hz, {tau_str}, Q_profile={q_prof_str} | "
        f"f_dft={r['f_dft']:.6f} Hz, Q_nls={q_nls_str}"
    )


In [ ]:
all_results = {}

for entry in RINGDOWN_FILES:
    path = data_dir / entry["name"]
    label = entry["label"]
    print(f"\n=== {label} ({path.name}) ===")

    pm = MokuPhasemeterObject(
        filename=str(path),
        start_time=entry["start_time"],
        duration=ANALYSIS_DURATION_S,
    )
    print(f"fs = {pm.fs:.1f} Hz, loaded duration = {pm.duration:.1f} s, nchan = {pm.nchan}")

    df = add_differential_channels(pm.df)
    t_rel = df["time"].values - df["time"].values[0]

    # Overview: differential channels
    fig, ax = plt.subplots(figsize=(12, 4), dpi=150)
    step = max(1, len(df) // 20000)
    for ch_name, _, _ in RINGDOWN_CHANNELS:
        ax.plot(t_rel[::step], df[ch_name].values[::step], label=ch_name, alpha=0.85)
    ax.set_xlabel("Time since segment start (s)")
    ax.set_ylabel("Differential phase (cycles)")
    ax.set_title(f"{label} – differential phase")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    file_results = {}
    for ch_name, col_a, col_b in RINGDOWN_CHANNELS:
        phase_cycles = df[ch_name].values
        r = analyzer.analyze_array(t=t_rel, data=phase_cycles, **NOISY_FIT_PARAMS)
        r["channel"] = ch_name
        r["file"] = entry["name"]
        file_results[ch_name] = r
        print(f"  {ch_name}: {format_result_summary(r)}")

    all_results[entry["name"]] = file_results

    fig, axes = plt.subplots(len(file_results), 1, figsize=(12, 4 * len(file_results)), squeeze=False)
    for idx, (ch_name, r) in enumerate(file_results.items()):
        ax = axes[idx, 0]
        step = max(1, len(r["t"]) // 50000)
        ax.plot(r["t"][::step], r["data"][::step], label="Data")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Phase (cycles)")
        ax.set_title(f"{label} / {ch_name}: {format_result_summary(r)}")
        ax.legend()
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
# Summary table across files and channels
rows = []
for fname, file_results in all_results.items():
    for ch_name, r in file_results.items():
        rows.append(
            {
                "file": fname,
                "channel": ch_name,
                "f_nls_Hz": r["f_nls"],
                "f_dft_Hz": r["f_dft"],
                "tau_profile_s": r.get("tau_profile"),
                "Q_profile": r.get("Q_profile"),
                "Q_nls": r.get("Q_nls"),
                "Q_profile_status": r.get("Q_profile_status"),
            }
        )

import pandas as pd

summary = pd.DataFrame(rows)
summary
